# NPTEL Study Notes: Introduction to Large Language Models (LLMs)

## Topic: Word Representations – From Count-based Methods to GloVe Embeddings

Welcome to your study notes! These notes are curated to help you master the mathematical and conceptual foundations of word representations, specifically bridging the gap between traditional methods and the **GloVe (Global Vectors)** embedding model. Pay special attention to the formulas, comparative advantages, and the "ice/steam" probability ratio, as these are prime targets for NPTEL MCQs and numerical questions.

---

### 1. Recap: Count-Based vs. Prediction-Based Methods

Before diving into GloVe, it is crucial to understand the limitations of previous approaches. GloVe was designed specifically to combine the best properties of both count-based and prediction-based methods.

#### A. Count-Based Methods (e.g., TF-IDF, PMI, PPMI)

These methods rely on building a large Co-occurrence Matrix (Term-Context Matrix) over the entire corpus.

* **Advantages:**
* **Fast to train:** You only need to scan the document corpus once to build the matrix.
* **Excellent Use of Statistics:** They effectively capture global statistical co-occurrence counts.


* **Disadvantages:**
* **Disproportionate Weighting:** They give too much importance to highly frequent, uninformative co-occurrences (e.g., stop words like "the", "and" appearing together).
* **Poor for Dense Embeddings:** Primarily used to capture raw similarity rather than producing the rich, dense vector spaces required for complex deep learning tasks.



#### B. Prediction-Based Methods (e.g., Word2Vec)

These methods learn embeddings by predicting target words from context words (or vice versa) using neural networks (like logistic regression applied over sliding windows).

* **Advantages:**
* **Captures Complex Patterns:** Excellent at capturing linear substructures and analogical relationships (e.g., *King - Man + Woman = Queen*).
* **Better Downstream Performance:** The dense embeddings generated rarely overfit and perform well on tasks like Sentiment Analysis and Named Entity Recognition (NER).


* **Disadvantages:**
* **Scales Poorly with Corpus Size:** If you add new documents to a fixed vocabulary, count-based metrics just update matrix counts. Prediction-based methods must re-scan the *entire* new text, sliding the context window and re-running the prediction algorithm (e.g., logistic regression) repeatedly for every window.
* **Ignores Global Statistics:** Because they train on local context windows sequentially, they do not leverage global co-occurrence statistics efficiently.



---

### 2. GloVe: Global Vectors for Word Representation

**Proposed in 2014 by Chris Manning’s group (Jeff Pennington, Richard Socher, Christopher Manning)**, GloVe combines the global statistical utilization of count-based models with the complex pattern-capturing abilities of prediction-based models.

#### A. Core Intuition: Co-occurrence Probabilities

Instead of looking at raw co-occurrence counts, GloVe looks at the **ratio of co-occurrence probabilities**.

Let $P(k\vert{}i)$ be the probability that a context word (pivot word) $k$ appears given a target word $i$.
Mathematically, $P(k\vert{}i) = \frac{X_{ik}}{X_i}$
Where:

* $X_{ik}$ = Number of times word $k$ appears in the context of word $i$.
* $X_i$ = Total number of times word $i$ appears in the corpus.

**The "Ice and Steam" Example:**
Consider target words $i = \text{ice}$ and $j = \text{steam}$. Let's evaluate how different pivot words ($k$) help differentiate them by looking at the ratio $\frac{P(k\vert{}i)}{P(k\vert{}j)}$:

| Pivot Word ($k$) | Context relevance | $P(k \vert{} \text{ice})$ | $P(k \vert{} \text{steam})$ | Ratio $\frac{P(k\vert{}\text{ice})}{P(k\vert{}\text{steam})}$ |
| --- | --- | --- | --- | --- |
| **Solid** | Relevant to ice, not steam | High | Low | **Large ($\gg 1$)** |
| **Gas** | Relevant to steam, not ice | Low | High | **Small ($\ll 1$)** |
| **Water** | Relevant to BOTH | High | High | **$\approx 1$** |
| **Fashion** | Relevant to NEITHER | Low | Low | **$\approx 1$** |

> **Exam Tip:** The probability ratio successfully distinguishes *relevant* discriminative words (solid, gas) from non-discriminative words (water, fashion). GloVe hypothesizes that word meanings are best captured by these ratios rather than raw probabilities.

#### B. Mathematical Derivation of the GloVe Objective

GloVe attempts to map the dot product of two word vectors to this statistical property.
Let $w_i$ and $w_j$ be the embeddings for words $i$ and $j$.

1. We want the dot product to model the log probability of co-occurrence:

$$w_i^T w_j \approx \log(X_{ij})$$


2. Because the relationship between a target and context word can be asymmetric in raw counts, the mathematical formulation derived in the lecture averages the probabilities, resulting in:

$$w_i^T w_j = \log(X_{ij}) - \frac{1}{2}\log(X_i) - \frac{1}{2}\log(X_j)$$


3. The terms $\frac{1}{2}\log(X_i)$ and $\frac{1}{2}\log(X_j)$ are dependent only on specific single words, not their co-occurrence. Therefore, GloVe absorbs them into **bias terms** ($b_i$ for the target word, $b_j$ for the context word):

$$w_i^T w_j + b_i + b_j = \log(X_{ij})$$



#### C. The Objective Function (Loss Function)

To learn the vectors $w$ and biases $b$, GloVe minimizes the squared difference between the dot product prediction and the actual log co-occurrence count.

**The Initial Loss:**


$$J = \sum_{i,j} (w_i^T w_j + b_i + b_j - \log X_{ij})^2$$

**The Weighting Function $f(X_{ij})$:**
To solve the count-based problem of giving disproportionate advantage to highly frequent word pairs (like stop words), GloVe introduces a weighting function $f(X_{ij})$ to the loss equation.

**Final Objective Function:**


$$J = \sum_{i,j} f(X_{ij}) (w_i^T w_j + b_i + b_j - \log X_{ij})^2$$

**Properties of $f(X_{ij})$:**
For the model to work correctly, $f(x)$ must satisfy three strict properties:

1. **$f(0) = 0$**: If two words never co-occur, the function must go to zero (finite continuous limit) so $\log(0)$ doesn't break the math.
2. **Non-decreasing**: It should never decrease as co-occurrence counts increase.
3. **Saturates for large values**: For very large counts (stop words), the function must cap out (be relatively small/flat) so it doesn't give them disproportionate weight.

*Mechanism:* GloVe uses a cutoff threshold ($x_{max}$). Up to $x_{max}$, the function grows. For any count $> x_{max}$, the weight is capped at 1.0.

#### D. Key Advantages of GloVe

* Fast training and scalable to huge corpora.
* Shows excellent performance even when trained on **small corpora**.
* Generates two embedding matrices (Target and Context), effectively capturing the dual nature of word relationships.

---

### 3. Properties and Applications of Word Embeddings

Once embeddings (like Word2Vec or GloVe) are trained, they exhibit fascinating linguistic properties:

1. **Linear Analogies:** They can solve analogy tests using basic vector arithmetic.
* *Geography:* $\text{Beijing} : \text{China} :: \text{Delhi} : \text{India}$
* *Syntax/Semantics:* $\text{King} - \text{Man} + \text{Woman} = \text{Queen}$


2. **Uncovering Bias:** Embeddings inherit human biases present in the training corpora.
* *Gender Bias Example:* $\text{Man} : \text{Doctor} :: \text{Woman} : \text{Nurse}$ or $\text{Father} : \text{Programmer} :: \text{Mother} : \text{Homemaker}$.


3. **Diachronic (Temporal) Studies:** By training separate embedding models on text from different decades (e.g., 1950s vs. 2010s New York Times articles), we can map how the meaning of words changes over time.
* *Example:* The word **"broadcast"** in the 1950s had neighbors related to agriculture (spreading seeds). In later decades, its neighbors shifted to radio and frequency-related terms.



---

### 4. NPTEL Practice Questions

**Q1. In the GloVe model, consider the target words $i = \text{ice}$ and $j = \text{steam}$. Let $k = \text{water}$ be a pivot word. Which of the following is true regarding the probability ratio $\frac{P(k\vert{}i)}{P(k\vert{}j)}$?**
A) The ratio will be significantly greater than 1.
B) The ratio will be significantly less than 1.
C) The ratio will be close to 1.
D) The ratio cannot be determined without the exact frequency counts.

* **Answer: C.** "Water" is equally relevant to both "ice" and "steam". Therefore, its co-occurrence probability with both is high, yielding a ratio close to 1. This signals to the model that "water" is not a good discriminative word for telling ice and steam apart.

**Q2. Which of the following correctly describes a primary disadvantage of prediction-based methods (like Word2Vec) compared to count-based methods?**
A) They fail to capture complex analogy patterns like King - Man + Woman.
B) They give too much weight to highly frequent stop-word pairs.
C) Training time scales poorly when new documents are added to the corpus.
D) They primarily evaluate raw similarity without generating dense vectors.

* **Answer: C.** Prediction-based methods require sliding a context window across every word of the new documents and repeatedly running logistic regression, making them scale poorly as the corpus grows. Options A, B, and D are incorrect (A and D describe strengths of prediction methods, B is a flaw of count-based methods).

**Q3. What is the purpose of the function $f(X_{ij})$ in the GloVe objective function?**
A) To convert the log probabilities back into raw counts.
B) To impose a penalty on highly frequent co-occurrences so they don't dominate the loss.
C) To ensure that stop words are given the highest possible gradient updates.
D) To handle negative co-occurrence values.

* **Answer: B.** The function $f(x)$ is designed to cap out at a maximum value ($x_{max}$) to prevent disproportionately high co-occurrences (like pairs of stop words) from overwhelming the loss function.

**Q4. According to the GloVe mathematical derivation, what is the role of the terms $b_i$ and $b_j$?**
A) They act as bias terms to replace the word-specific marginal probability components $\log(X_i)$ and $\log(X_j)$.
B) They represent the temporal shift of words across different decades.
C) They are regularizers applied to the weighting function $f(X_{ij})$.
D) They represent the raw frequency of the pivot word $k$.

* **Answer: A.** In the derivation, the components that depend only on the individual words (not their co-occurrence) are mapped to trainable bias terms for the target and context words, leading to the formulation $w_i^T w_j + b_i + b_j = \log(X_{ij})$.

---

### 5. Quick Revision Summary

* **Count-based methods** (TF-IDF, PMI) use global statistics well but handle frequent words poorly and lack dense representations.
* **Prediction-based methods** (Word2Vec) create excellent dense representations that capture complex analogies but scale poorly with corpus size.
* **GloVe (Global Vectors)** combines both by training on a co-occurrence matrix instead of sliding windows, ensuring fast training and scalability.
* GloVe relies on the **ratio of co-occurrence probabilities**, which excellently separates discriminative context words from generic ones.
* The GloVe objective function is $J = \sum f(X_{ij}) (w_i^T w_j + b_i + b_j - \log X_{ij})^2$.
* The **weighting function $f(X_{ij})$** is strictly necessary to prevent frequent words (stop words) from dominating the loss.
* Word embeddings can be used for deep linguistic analysis, including solving analogies, identifying cultural biases, and mapping semantic shifts over time (diachronic analysis).